<center><h1>Gong__jiale__HW3</h1></center>
<br>
<br>

Name: Jiale Gong
<br>
Github Username: jialegong
<br>
USC ID: 1387502580

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [100]:
import os
import io
import zipfile
import requests
from scipy.stats import bootstrap
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

Get the AReM Data Set

In [103]:
url = (
    "https://archive.ics.uci.edu/static/public/366/"
    "activity+recognition+system+based+on+multisensor+"
    "data+fusion+arem.zip"
)
ALL_COLUMNS = ["time", "avg_rss12", "var_rss12", "avg_rss13",
               "var_rss13", "avg_rss23", "var_rss23"]
SERIES_NAMES = ["avg_rss12", "var_rss12", "avg_rss13",
                "var_rss13", "avg_rss23", "var_rss23"]
ACTIVITIES = ["bending1", "bending2", "cycling", "lying",
              "sitting", "standing", "walking"]


print("Downloading AReM Dataset")

response = requests.get(url)
response.raise_for_status()
print("Status code:", response.status_code)
print("Downloaded file size:", len(response.content), "bytes")

def parse_arem_csv(csv_bytes, expected_rows=480):
    text = csv_bytes.decode("utf-8", errors="replace")
    data_rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        line = line.rstrip(",").strip()
        if "," in line:
            fields = [f.strip() for f in line.split(",")]
        else:
            fields = line.split()
        fields = [f for f in fields if f != ""]
        if len(fields) != len(ALL_COLUMNS):
            continue
        data_rows.append(fields)

    if len(data_rows) == 0:
        raise ValueError("No valid data rows were found")

    df = pd.DataFrame(data_rows, columns=ALL_COLUMNS)
    df = df.apply(pd.to_numeric, errors="coerce")

    if df.isna().any().any():
        raise ValueError("Non-numeric values remain after cleaning")

    if len(df) < expected_rows:
        df = df.reindex(range(expected_rows)).ffill().bfill()
    elif len(df) > expected_rows:
        df = df.iloc[:expected_rows].reset_index(drop=True)

    return df

instances = []
with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    for name in archive.namelist():
        if not name.endswith(".csv"):
            continue
        folder = name.split("/")[0]
        if folder not in ACTIVITIES:
            continue
        df = parse_arem_csv(archive.read(name))
        instances.append((folder, name, df))

print("\nTotal instances:", len(instances))
print("Shapes found:", {df.shape for _, _, df in instances})

counts = Counter(label for label, _, _ in instances)
print("\nInstances per activity:")
for activity in ACTIVITIES:
    print(f"  {activity:<10}: {counts[activity]}")

label, name, df = instances[0]
print(f"\nExample: {name}")
print(f"Shape: {df.shape}")
print(df.head().to_string(index=False))

cleaning_summary = pd.DataFrame({
    "File": ["bending2/dataset4.csv", "cycling/dataset9.csv",
             "cycling/dataset14.csv", "sitting/dataset8.csv"],
    "Problem": ["Whitespace-separated instead of comma-separated",
                "Extra trailing comma", "Extra trailing comma",
                "Only 479 rows"],
    "Cleaning Method": ["Split on whitespace", "Remove trailing comma",
                        "Remove trailing comma", "Forward-fill to 480 rows"],
})
print("\nData Cleaning Summary:")
print(cleaning_summary.to_string(index=False))

Status code: 200
Downloaded file size: 524200 bytes

Total instances: 88
Shapes found: {(480, 7)}

Instances per activity:
  bending1  : 7
  bending2  : 6
  cycling   : 15
  lying     : 15
  sitting   : 15
  standing  : 15
  walking   : 15

Example: bending1/dataset1.csv
Shape: (480, 7)
 time  avg_rss12  var_rss12  avg_rss13  var_rss13  avg_rss23  var_rss23
    0      39.25       0.43      22.75       0.43      33.75        1.3
  250      39.25       0.43      23.00       0.00      33.00        0.0
  500      39.25       0.43      23.25       0.43      33.00        0.0
  750      39.50       0.50      23.00       0.71      33.00        0.0
 1000      39.50       0.50      24.00       0.00      33.00        0.0

Data Cleaning Summary:
                 File                                         Problem          Cleaning Method
bending2/dataset4.csv Whitespace-separated instead of comma-separated      Split on whitespace
 cycling/dataset9.csv                            Extra trailing co

In [105]:
def parse_strict_csv(csv_bytes, expected_rows=480):
    text = csv_bytes.decode("utf-8", errors="replace")
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        fields = [f.strip() for f in line.split(",")]
        if len(fields) != len(ALL_COLUMNS) or any(f == "" for f in fields):
            raise ValueError("Invalid fields")
        rows.append(fields)
    if len(rows) != expected_rows:
        raise ValueError("Wrong row count")
    df = pd.DataFrame(rows, columns=ALL_COLUMNS)
    return df.apply(pd.to_numeric, errors="raise")

instances_skipped, skipped_files = [], []
with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    for name in archive.namelist():
        if not name.endswith(".csv"):
            continue
        folder = name.split("/")[0]
        if folder not in ACTIVITIES:
            continue
        try:
            df_s = parse_strict_csv(archive.read(name))
            instances_skipped.append((folder, name, df_s))
        except (ValueError, TypeError):
            skipped_files.append(name)

print("\nSkipped files:", skipped_files)
print("Instances without cleaning:", len(instances_skipped))

def split(inst_list):
    train, test = [], []
    for label, fname, df in inst_list:
        n = int(os.path.basename(fname).replace("dataset", "").replace(".csv", ""))
        is_test = (n in [1, 2]) if label in ["bending1", "bending2"] else (n in [1, 2, 3])
        (test if is_test else train).append((label, fname, df))
    return train, test

def feats(df):
    v = []
    for c in SERIES_NAMES:
        s = df[c].to_numpy()
        v += [np.min(s), np.max(s), np.mean(s), np.median(s),
              np.std(s, ddof=1), np.percentile(s, 25), np.percentile(s, 75)]
    return v

def to_arrays(inst_list):
    X, y = [], []
    for label, _, df in inst_list:
        X.append(feats(df)); y.append(label)
    return np.array(X), np.array(y)

tr_A, te_A = split(instances_skipped)
XA_tr, yA_tr = to_arrays(tr_A); XA_te, yA_te = to_arrays(te_A)
rf_A = RandomForestClassifier(n_estimators=200, random_state=42).fit(XA_tr, yA_tr)
acc_A = accuracy_score(yA_te, rf_A.predict(XA_te))

tr_B, te_B = split(instances)
XB_tr, yB_tr = to_arrays(tr_B); XB_te, yB_te = to_arrays(te_B)
rf_B = RandomForestClassifier(n_estimators=200, random_state=42).fit(XB_tr, yB_tr)
acc_B = accuracy_score(yB_te, rf_B.predict(XB_te))

print("\nImpact of Data Cleaning")
print(f"  A: Skip problematic files: {len(instances_skipped)} instances | "
      f"Train {len(tr_A)} / Test {len(te_A)} | Acc = {acc_A:.4f}")
print(f"  B: Clean all files:        {len(instances)} instances | "
      f"Train {len(tr_B)} / Test {len(te_B)} | Acc = {acc_B:.4f}")
print(f"  Difference: {acc_B - acc_A:+.4f}")


Skipped files: ['bending2/dataset4.csv', 'cycling/dataset14.csv', 'cycling/dataset9.csv', 'sitting/dataset8.csv']
Instances without cleaning: 84

Impact of Data Cleaning
  A: Skip problematic files: 84 instances | Train 65 / Test 19 | Acc = 0.8947
  B: Clean all files:        88 instances | Train 69 / Test 19 | Acc = 0.8947
  Difference: +0.0000


Impact of Cleaning. Two settings were compared using the same
Random Forest procedure:

| Setting                     | Instances | Train | Test | Test Accuracy |
|-----------------------------|----------:|------:|-----:|--------------:|
| A: Skip problematic files   |        84 |    65 |   19 |        0.8947 |
| B: Clean and retain all     |        88 |    69 |   19 |        0.8947 |
| Difference                  |        +4 |    +4 |   0  |        0.0000 |

Under strict parsing, four files were skipped:
`bending2/dataset4.csv`, `cycling/dataset9.csv`,
`cycling/dataset14.csv`, and `sitting/dataset8.csv`. All four belong to
the training portion of the split (since test instances are datasets 1–2
for bending activities and datasets 1–3 for the remaining activities),
so cleaning increased the training set from 65 to 69 observations while
the test set remained unchanged at 19.

The test accuracy was unchanged at 0.8947 (17 out of 19 correct). For
this particular train/test split and Random Forest model, the four
recovered training instances did not change the model's predictions on
the test set.

Cleaning was nevertheless retained because it:
1. preserves all 88 available observations,
2. produces consistent `(480, 7)` input files across all instances, and
3. provides a reproducible way to handle the documented formatting
   irregularities in the dataset.

### (b) Test and Train Data

In [55]:
train_instances = []
test_instances = []

for label, fname, df in instances:
    base = os.path.basename(fname)                              # e.g. "dataset3.csv"
    dataset_num = int(base.replace("dataset", "").replace(".csv", ""))

    if label in ["bending1", "bending2"]:
        is_test = dataset_num in [1, 2]
    else:
        is_test = dataset_num in [1, 2, 3]

    if is_test:
        test_instances.append((label, fname, df))
    else:
        train_instances.append((label, fname, df))

print(f"Training instances: {len(train_instances)}")
print(f"Test instances:     {len(test_instances)}")

train_counts = Counter(label for label, _, _ in train_instances)
test_counts  = Counter(label for label, _, _ in test_instances)

print("\nTrain/Test distribution per activity:")
print(f"{'Activity':<10} {'Train':>6} {'Test':>6}")
print("-" * 26)
for activity in ACTIVITIES:
    print(f"{activity:<10} {train_counts[activity]:>6} {test_counts[activity]:>6}")
print("-" * 26)
print(f"{'Total':<10} {len(train_instances):>6} {len(test_instances):>6}")

Training instances: 69
Test instances:     19

Train/Test distribution per activity:
Activity    Train   Test
--------------------------
bending1        5      2
bending2        4      2
cycling        12      3
lying          12      3
sitting        12      3
standing       12      3
walking        12      3
--------------------------
Total          69     19


Following the assignment instructions, the dataset was split as follows. The test set consists of datasets 1 and 2 from bending1 and bending2, and datasets 1, 2, and 3 from all other activity folders. All remaining instances form the training set.

This results in 69 training instances and 19 test instances. The per-activity breakdown is: bending1 has 5 training and 2 test instances; bending2 has 4 training and 2 test instances; and each of cycling, lying, sitting, standing, and walking has 12 training and 3 test instances.

The split follows the rule exactly: for bending1 and bending2, only datasets 1 and 2 are used for testing, while for all other activities datasets 1, 2, and 3 are used for testing. All remaining datasets form the training set.

Note that all three files that required cleaning (cycling/dataset9.csv, cycling/dataset14.csv, and sitting/dataset8.csv) belong to the training portion of the split, since test instances for cycling and sitting are datasets 1–3. Therefore, cleaning increases the number of training instances from 66 to 69 without affecting the test set.



### (c) Feature Extraction

#### i. Research

Common time-domain features include basic statistics (min, max, mean, median, std, variance, quartiles, skewness, kurtosis), waveform/energy features (peak-to-peak, RMS, zero-crossing rate, autocorrelation), and trend/model features (AR coefficients, correlation).

In this assignment, seven features are used: minimum, maximum, mean, median, standard deviation, first quartile, and third quartile.

#### ii. Extraction

In [124]:
SERIES_NAMES = ["avg_rss12", "var_rss12", "avg_rss13",
                "var_rss13", "avg_rss23", "var_rss23"]

FEATURE_NAMES = ["min", "max", "mean", "median", "std", "q1", "q3"]


def extract_features(series):
    """Extract 7 time-domain features from one time series."""
    return [
        np.min(series),
        np.max(series),
        np.mean(series),
        np.median(series),
        np.std(series, ddof=1),
        np.percentile(series, 25),
        np.percentile(series, 75),
    ]


def instance_to_vector(df):
    """Convert one instance (6 series) into a 42-dim feature vector."""
    vec = []
    for col in SERIES_NAMES:
        vec.extend(extract_features(df[col].values))
    return vec


feature_col_names = [f"{col}_{feat}"
                     for col in SERIES_NAMES
                     for feat in FEATURE_NAMES]


def build_dataset(instance_list):
    X, y = [], []
    for label, fname, df in instance_list:
        X.append(instance_to_vector(df))
        y.append(label)
    return np.array(X), np.array(y)


X_train, y_train = build_dataset(train_instances)
X_test,  y_test  = build_dataset(test_instances)

print("Feature Extraction")

print(f"Training feature matrix: {X_train.shape}")
print(f"Test feature matrix:     {X_test.shape}")


X_all, y_all = build_dataset(instances)

df_all_features = pd.DataFrame(X_all, columns=feature_col_names)

instance_names = [
    f"{label}/{os.path.basename(fname)}"
    for label, fname, _ in instances
]

df_all_features.insert(0, "instance", instance_names)
df_all_features.insert(1, "label", y_all)

print("\nComplete numerical feature matrix:", X_all.shape)
print("Complete dataset including instance and label:",
      df_all_features.shape)

print("\nFirst 5 rows (first 8 columns):")
print(df_all_features.iloc[:5, :8].to_string(index=False))

Feature Extraction
Training feature matrix: (69, 42)
Test feature matrix:     (19, 42)

Complete numerical feature matrix: (88, 42)
Complete dataset including instance and label: (88, 44)

First 5 rows (first 8 columns):
             instance    label  avg_rss12_min  avg_rss12_max  avg_rss12_mean  avg_rss12_median  avg_rss12_std  avg_rss12_q1
bending1/dataset1.csv bending1          37.25          45.00       40.624792             40.50       1.476967         39.25
bending1/dataset2.csv bending1          38.00          45.67       42.812812             42.50       1.435550         42.00
bending1/dataset3.csv bending1          35.00          47.40       43.954500             44.33       1.558835         43.00
bending1/dataset4.csv bending1          33.00          47.75       42.179812             43.50       3.670666         39.15
bending1/dataset5.csv bending1          33.00          45.75       41.678063             41.75       2.243490         41.33


#### iii. Standard Deviation

In [130]:
feature_stds = np.std(X_all, axis=0, ddof=1)


def sample_std(x, axis=-1):
    """Sample standard deviation used in bootstrap."""
    return np.std(x, axis=axis, ddof=1)


ci_results = []

N_RESAMPLES = 1000
CONFIDENCE = 0.90

for j, feature_name in enumerate(feature_col_names):
    data = X_all[:, j]
    std_est = feature_stds[j]

    # Handle features that are constant across all instances
    if np.isclose(std_est, 0.0):
        ci_results.append({
            "feature": feature_name,
            "std_estimate": 0.0,
            "ci_low": 0.0,
            "ci_high": 0.0,
            "ci_width": 0.0,
            "note": "degenerate"
        })
        continue

    result = bootstrap(
        (data,),
        sample_std,
        vectorized=True,
        confidence_level=CONFIDENCE,
        n_resamples=N_RESAMPLES,
        method="BCa",
        random_state=42
    )

    ci_low = result.confidence_interval.low
    ci_high = result.confidence_interval.high

    ci_results.append({
        "feature": feature_name,
        "std_estimate": std_est,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "ci_width": ci_high - ci_low,
        "note": ""
    })


ci_df = pd.DataFrame(ci_results)

# Display all 42 features
display(ci_df.round(4))

degenerate_features = ci_df.loc[
    ci_df["note"] == "degenerate", "feature"
].tolist()

print("Number of features:", len(ci_df))
print("Degenerate features:", degenerate_features)

ci_df.to_csv("arem_bootstrap_ci.csv", index=False)

,feature,std_estimate,ci_low,ci_high,ci_width,note
0,avg_rss12_min,9.5700,8.5377,10.9852,2.4475,
1,avg_rss12_max,4.3944,3.5587,5.5561,1.9974,
2,avg_rss12_mean,5.3357,4.8514,6.0371,1.1857,
3,avg_rss12_median,5.4401,4.9311,6.1634,1.2323,
4,avg_rss12_std,1.7722,1.6030,1.9943,0.3912,
5,avg_rss12_q1,6.1536,5.6888,6.7789,1.0901,
6,avg_rss12_q3,5.1389,4.4563,5.9679,1.5116,
7,var_rss12_min,0.0000,0.0000,0.0000,0.0000,degenerate
8,var_rss12_max,5.0627,4.7080,5.4671,0.7591,
9,var_rss12_mean,1.5742,1.4224,1.7166,0.2943,


Number of features: 42
Degenerate features: ['var_rss12_min', 'var_rss13_min']


For each of the 42 extracted features, the sample standard deviation was
calculated across all 88 instances using ddof=1. A 90% bootstrap
confidence interval was then constructed using the BCa method with 1,000
resamples and a fixed random seed for reproducibility.

Two features, `var_rss12_min` and `var_rss13_min`, are degenerate because
they take the value 0 in every instance. Their sample standard deviations
are therefore exactly 0, and their confidence intervals are trivially
[0, 0]. The remaining 40 features have non-degenerate bootstrap
confidence intervals. The complete results for all 42 features are shown
in the table above.

#### iv. Select Features

In [133]:
selected_feature_types = [
    "mean",
    "std",
    "max"
]

print("Selected time-domain feature types:")

for i, feature in enumerate(
    selected_feature_types,
    start=1
):
    print(f"{i}. {feature}")


selected_feature_columns = [
    f"{series}_{feature}"
    for series in SERIES_NAMES
    for feature in selected_feature_types
]

selected_feature_indices = [
    feature_col_names.index(column)
    for column in selected_feature_columns
]


X_all, y_all = build_dataset(instances)

X_selected = X_all[
    :,
    selected_feature_indices
]

df_selected_features = pd.DataFrame(
    X_selected,
    columns=selected_feature_columns
)

instance_names = [
    f"{label}/{os.path.basename(file_name)}"
    for label, file_name, _ in instances
]

df_selected_features.insert(
    0,
    "instance",
    instance_names
)

df_selected_features.insert(
    1,
    "label",
    y_all
)



print(
    "\nNumber of selected feature types:",
    len(selected_feature_types)
)

print(
    "Number of selected numerical features:",
    X_selected.shape[1]
)

print(
    "Shape of selected feature matrix:",
    X_selected.shape
)

display(df_selected_features.head())

Selected time-domain feature types:
1. mean
2. std
3. max

Number of selected feature types: 3
Number of selected numerical features: 18
Shape of selected feature matrix: (88, 18)


,instance,label,avg_rss12_mean,avg_rss12_std,avg_rss12_max,var_rss12_mean,var_rss12_std,var_rss12_max,avg_rss13_mean,avg_rss13_std,avg_rss13_max,var_rss13_mean,var_rss13_std,var_rss13_max,avg_rss23_mean,avg_rss23_std,avg_rss23_max,var_rss23_mean,var_rss23_std,var_rss23_max
0,bending1/dataset1.csv,bending1,40.624792,1.476967,45.00,0.358604,0.322605,1.30,19.040937,4.462952,29.50,0.832542,0.965659,7.23,34.311292,2.188449,38.25,0.570583,0.582915,1.92
1,bending1/dataset2.csv,bending1,42.812812,1.435550,45.67,0.372437,0.289158,1.22,20.096979,3.893737,29.50,0.876438,1.047259,5.76,33.024583,1.995255,38.50,0.571083,0.601010,3.11
2,bending1/dataset3.csv,bending1,43.954500,1.558835,47.40,0.426250,0.338690,1.70,22.122354,3.030943,29.75,0.497312,0.550657,4.44,35.588458,1.999604,38.50,0.493292,0.513506,1.79
3,bending1/dataset4.csv,bending1,42.179812,3.670666,47.75,0.696042,0.630860,3.00,22.183625,3.810469,30.00,0.989917,0.953730,5.15,33.493917,3.849448,38.67,0.613521,0.524317,2.18
4,bending1/dataset5.csv,bending1,41.678063,2.243490,45.75,0.535979,0.405469,2.83,19.006563,4.087107,28.25,0.841875,0.928801,6.42,29.857083,2.411026,37.50,0.383292,0.389164,1.79


I select mean, standard deviation, and maximum as the three most important time-domain feature types. Mean represents the central tendency of a signal and can distinguish activities with different average RSS levels. Standard deviation measures signal variability and helps differentiate dynamic activities, such as walking and cycling, from relatively static activities, such as sitting and lying. Maximum captures peak signal intensity and can identify activities that produce unusually large RSS values.
These three features provide complementary information about the center, variability, and upper extreme of each time series. Since each of the three selected feature types is extracted from all six time series, the selected dataset contains 6 × 3 = 18 numerical features per instance.

## 2. ISLR 3.7.4

### (a) Linear Train

We expect the cubic regression to have a lower (or equal) training RSS.
The training RSS measures how well the model fits the training data. The ordinary least squares (OLS) method explicitly minimizes the training RSS. The cubic model is a more flexible model that encompasses the linear model as a special case (specifically, when 
β2=0 and β3=0). Because the cubic model has a larger hypothesis space, it can perfectly mimic the linear model or fit the training data even more closely by capturing random noise (overfitting). Therefore, the minimum achievable training RSS for the cubic regression will always be less than or equal to that of the linear regression, regardless of the true underlying relationship.

### (b) Linear Test

We expect the linear regression to have a lower test RSS.
Since the true relationship is linear, the linear model is correctly specified and has zero bias. The cubic model, being more flexible, will likely overfit the training data by fitting the noise. This leads to high variance when predicting new, unseen data. Due to the bias-variance tradeoff, the simpler linear model will generalize better to the test set, resulting in a lower test RSS compared to the overly complex cubic model.

### (c) Not Linear Train

We expect the cubic regression to have a lower (or equal) training RSS.
This is the same logic as part (a). Regardless of the true relationship, the cubic model's hypothesis space contains the linear model's hypothesis space. Because OLS minimizes training RSS, a more flexible model will always fit the training data at least as well as (and usually better than) a less flexible one. Therefore, the cubic regression will always have a lower or equal training RSS.

### (d) Not Linear Testing

There is not enough information to tell.
The test RSS depends on the bias-variance tradeoff, which is determined by how far the true relationship deviates from linear and the amount of noise in the data.

· If the true relationship is only slightly non-linear, the linear model might still yield a lower test RSS because the cubic model's high variance (from overfitting) outweighs its reduction in bias.

· If the true relationship is highly non-linear, the cubic model will likely yield a lower test RSS because the linear model's high bias (underfitting) will dominate.

· Without knowing the exact shape of the true function or the noise variance, we cannot theoretically determine which model will have a lower test RSS.



## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 